# Results  

Final results from the pipeline are stored in subdirectory *plateanalysis/footprints/results/*. 

Each file contains the result of a pipeline run over a *sequence* of plates. Sequences of plates are defined in file *settings.py*, and created based on the output of notebook *footprints.ipynb*.

The result file contains just the very final product; intermediate results are not stored in GitHub due to the policy of not storing there files that are either temporary, or potentially large, such as FITS binary tables and such. Intermediate results can be seen only by running the code locally over the appropriate data sets.  

#### Note

Due to sheer size, only one of the plate sequences I worked with has its pipeline output uploaded to the repo. Also, GitHub is unable to display the html due to its size. Download the file and use your browser locally to display the file.  

Alternatively, you can download the result files from Google Drive: https://drive.google.com/drive/folders/164xc4faMDbPt84ZileJZoHzK94kvG8Gm


Below is a summary of findings to date. Numbers refer to the plate ID code in the APPLAUSE archive.

- Data from the **1m-Spiegelteleskop** and the **Doppel-Reflektor** telescopes: still to be reviewed in light of the upgraded software.

- Data from the **Grosser Schmidt-Spiegel** telescope yeld a few candidates.




### Notes

APPLAUSE scans were performed on the original plates, not copies.

## Processing stats

Builds a summary of number of detections/objects for each plate pair.

In [1]:
import os
import warnings

from astropy.table import Table
from astropy.time import Time
from astropy.utils.exceptions import AstropyWarning

import pandas as pd
from erfa import ErfaWarning

from library import get_earth_shadow
import settings
from settings import CATALOG, get_parameters, fname, sequences

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=ErfaWarning)
warnings.simplefilter('ignore', category=FutureWarning)

In [3]:
def make_table():
    table = Table(names=['seq', 'plate 1', 'ut_mid_1', 'exptime_1', 'es_1',
                                'plate 2', 'ut_mid_2', 'exptime_2', 'es_2',
                         'total sources', 'matched', 'non-matched', 'candidates'],
                  dtype=['S1', 'S1', 'S1', 'i4', 'f4',
                               'S1', 'S1', 'i4', 'f4', 
                         'i4', 'i4', 'i4', 'i4'])

    return table

In [4]:
# use Pandas to display table so all rows can be displayed at once. Displaying long
# Astropy tables directly causes only the first and last few rows to be displayed. 

def display_with_pandas(table):
    df = table.to_pandas()

    # avoid display of unicode reminder in every string field
    df['seq'] = df['seq'].apply(lambda x: x.decode('utf-8'))
    df['plate 1'] = df['plate 1'].apply(lambda x: x.decode('utf-8'))
    df['plate 2'] = df['plate 2'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_1'] = df['ut_mid_1'].apply(lambda x: x.decode('utf-8'))
    df['ut_mid_2'] = df['ut_mid_2'].apply(lambda x: x.decode('utf-8'))

    # style with CSS properties for table headers (th) and data cells (td)
    th_props = [('font-size', '12px'), ('text-align', 'right'), ('font-weight', 'bold')]
    td_props = [('font-size', '11px'), ('text-align', 'right')]
    styles = [dict(selector="th", props=th_props),dict(selector="td", props=td_props)]
    styled_df = df.style.set_table_styles(styles)    
    
    format_mapping = {
        'es_1': '{:.3}'.format,
        'es_2': '{:.3}'.format,
    }
    styled_df = styled_df.format(format_mapping)    

    display(styled_df)

In [5]:
# get metadata for a given plate

def get_metadata(plate, catalog_table):
    
    mask = catalog_table['plate_id'] == plate
    t = catalog_table[mask]
    
    long = t['site_longitude'][0]
    lat  = t['site_latitude'][0]
    ra  = t['ra_icrs'][0]
    dec = t['dec_icrs'][0]
    time_event = Time(t['ut_mid'][0])

    es, _ = get_earth_shadow(ra, dec, time_event, longitude=long, latitude=lat)
    
    return t['ut_mid'][0], t['exptime'][0], es

In [6]:
catalog_table = Table.read(CATALOG, format='ascii.csv')

result = make_table()

for seq_key in list(sequences.keys()):
    seq = sequences[seq_key]

    for i in range(len((seq)) - 1):
        try:
            plate_id = seq[i]
            next_plate_id = seq[i+1]
            
            # metadata
            ut_mid_1, exptime_1, es_1 = get_metadata(plate_id, catalog_table)
            ut_mid_2, exptime_2, es_2 = get_metadata(next_plate_id, catalog_table)
            
            plate_id_str = str(plate_id)
            next_plate_id_str = str(next_plate_id)

            key = plate_id_str + ',' + next_plate_id_str
            par = get_parameters(key)

            # row count on each relevant product table
            table_sources = Table.read(fname(par['table1']), format='ascii.csv')
            mask = table_sources['scan_id'] == table_sources['scan_id'][0]
            table_sources = table_sources[mask]
            sources = len(table_sources)

            table_matched = Table.read(fname(par['table_matched']), format='fits')
            matched = len(table_matched)

            table_non_matched = Table.read(fname(par['table_non_matched']), format='fits')
            non_matched = len(table_non_matched)

            table_candidates = Table.read(fname(par['table_candidates']), format='fits')
            candidates = len(table_candidates)

            result.add_row([seq_key, plate_id_str, ut_mid_1, exptime_1, es_1,
                            next_plate_id_str, ut_mid_2, exptime_2, es_2,
                            sources, matched, non_matched, candidates])
        except KeyError:
            break

### Summary table

Table columns:

- **seq**: sequence ID generated by notebook *footprints.ipynb*
- **plate ...**: plate IDs defining the pair of plates
- **ut_mid ...**: UT mid for each plate
- **exptime ...**: EXPTIME for each plate
- **es ...**: Earth's shadow angle (deg) for each plate
- **total sources**: total number of sources in the APPLAUSE table of the first plate, for the *_x* scan direction
- **matched**: number of sources in the first plate that have a match in the second plate, after the first plate's source table gets prunned of bad detections (as per sextractor-generated parameters)
- **non-matched**: number of sources in the first plate (after it was cleaned up as above) that have **NO** match on the second plate
- **candidates**: number of candidates remaining after filtering is applyied (filtering based on Gaussian fitting, radial profile, circularity, and aperture photometry analysis)


In [7]:
display_with_pandas(result)

,seq,plate 1,ut_mid_1,exptime_1,es_1,plate 2,ut_mid_2,exptime_2,es_2,total sources,matched,non-matched,candidates
0,seq01,9245,1956-09-25 19:00:47,1800,69.3,9246,1956-09-25 19:43:40,1800,68.4,590001,246737,27726,1
1,seq01,9246,1956-09-25 19:43:40,1800,68.4,9247,1956-09-25 20:29:33,1800,67.6,465264,233033,16063,16
2,seq03,9313,1956-12-03 17:15:31,900,62.9,9315,1956-12-03 18:06:23,900,61.8,55239,17234,93,0
3,seq03,9315,1956-12-03 18:06:23,900,61.8,9317,1956-12-03 18:56:48,900,60.6,76057,22462,2029,1
4,seq03,9317,1956-12-03 18:56:48,900,60.6,9318,1956-12-03 19:26:11,900,60.0,59981,21413,426,0
5,seq03,9318,1956-12-03 19:26:11,900,60.0,9319,1956-12-03 20:27:18,900,58.8,62314,21293,139,1
6,seq03,9319,1956-12-03 20:27:18,900,58.8,9320,1956-12-03 20:55:56,900,58.3,75926,23992,380,1
7,seq04,9322,1956-12-06 18:54:56,900,61.9,9323,1956-12-06 19:19:52,900,61.4,58894,20801,315,2
8,seq04,9323,1956-12-06 19:19:52,900,61.4,9324,1956-12-06 19:44:24,900,60.9,55417,21108,475,0
9,seq04,9324,1956-12-06 19:44:24,900,60.9,9325,1956-12-06 20:08:14,900,60.4,54232,18804,97,0


## To do

- fix *footprints.ipynb*: time order in sequences of plates.

- multivariate probability.

- add circularity diagnostic - DONE
- add other shape diagnostics - DONE
- profile metrics - DONE
- refactor towards batch processing - DONE
- automate data download - DONE
- add Earth shadow angle to tables - DONE

## Summary of possible bright event in plate 9319

### Needs revision: other similar events exist

- date 1956-12-03
- time between utc_mid: 28 min
- exposure time on each plate: 15 min
- emulsion Kodak Oa-E both plates

Plates 9313, 9315, 9316, 9317, 9318, were searched for similar events, with no detection. These plates were all taken on the same night of 1956-12-03 over a 4 hr time span, with the same telescope pointing (except 9316 that was taken 24 hr later). 

The event position on the plate (distance to center = 3248.3 px, distance to edge = 1061.8 px, annular bin = 4) places it in a region where artifacts can be perhaps more likely to happen. The profile suggests that it might be indeed an artifact: these tend to have systematically narrower widths than stars of same brightness.  

Plate's FOV is far from Earth's shadow. 